# W1-D1: Metric Anomaly Detection

Notebook phi?n b?n chu?n h?a logic theo h??ng production, b?m ??ng rubric assignment.

## 1. Imports & Config

In [1]:
from __future__ import annotations

import gzip
import json
import pickle
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf

plt.style.use("seaborn-v0_8-whitegrid")
try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)


In [2]:
@dataclass(frozen=True)
class ProjectConfig:
    root: Path
    data_dir: Path
    output_dir: Path
    model_dir: Path
    series_key: str = "realKnownCause/machine_temperature_system_failure.csv"


def resolve_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("Khong tim thay data/raw. Hay mo notebook trong D:/AWS/AIOPS")


ROOT = resolve_project_root()
CFG = ProjectConfig(
    root=ROOT,
    data_dir=ROOT / "data" / "raw",
    output_dir=ROOT / "artifacts" / "outputs",
    model_dir=ROOT / "artifacts" / "models",
)
CFG.output_dir.mkdir(parents=True, exist_ok=True)
CFG.model_dir.mkdir(parents=True, exist_ok=True)

print("ROOT:", CFG.root)
print("DATA:", CFG.data_dir)
print("OUT:", CFG.output_dir)
print("MODEL:", CFG.model_dir)

ROOT: D:\AWS\AIOPS
DATA: D:\AWS\AIOPS\data\raw
OUT: D:\AWS\AIOPS\artifacts\outputs
MODEL: D:\AWS\AIOPS\artifacts\models


## 2. Data Loading & Labeling

In [3]:
def load_primary_series(cfg: ProjectConfig) -> pd.DataFrame:
    csv_path = cfg.data_dir / "machine_temperature_system_failure.csv"
    label_path = cfg.data_dir / "combined_windows.json"

    if not csv_path.exists() or not label_path.exists():
        raise FileNotFoundError("Thieu file du lieu hoac labels")

    df = pd.read_csv(
        csv_path,
        usecols=["timestamp", "value"],
        parse_dates=["timestamp"],
        engine="python",
    ).sort_values("timestamp").reset_index(drop=True)

    with label_path.open("r", encoding="utf-8") as f:
        windows = json.load(f)[cfg.series_key]

    df["label"] = False
    for start, end in windows:
        mask = df["timestamp"].between(pd.Timestamp(start), pd.Timestamp(end), inclusive="both")
        df.loc[mask, "label"] = True

    return df


df = load_primary_series(CFG)
display(df.head())
print(df.shape, "anomaly points:", int(df["label"].sum()))

,timestamp,value,label
0,2013-12-02 21:15:00,73.967322,False
1,2013-12-02 21:20:00,74.935882,False
2,2013-12-02 21:25:00,76.124162,False
3,2013-12-02 21:30:00,78.140707,False
4,2013-12-02 21:35:00,79.329836,False


(22695, 3) anomaly points: 2268


## 3. EDA

In [4]:
def summarize_series(df: pd.DataFrame) -> pd.DataFrame:
    row = {
        "rows": len(df),
        "start": df["timestamp"].min(),
        "end": df["timestamp"].max(),
        "mean": df["value"].mean(),
        "std": df["value"].std(),
        "skewness": stats.skew(df["value"]),
        "min": df["value"].min(),
        "max": df["value"].max(),
        "labelled_anomaly_points": int(df["label"].sum()),
    }
    out = pd.DataFrame([row])
    out.to_csv(CFG.output_dir / "eda_stats.csv", index=False)
    return out


display(summarize_series(df))

,rows,start,end,mean,std,skewness,min,max,labelled_anomaly_points
0,22695,2013-12-02 21:15:00,2014-02-19 15:25:00,85.926498,13.746912,-1.833686,2.084721,108.510543,2268


In [6]:
def plot_eda(df: pd.DataFrame, out_dir: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("EDA - Machine Temperature System Failure", fontsize=16, fontweight="bold")

    axes[0, 0].hist(df["value"], bins=34, density=True, alpha=0.72, color="#5b5aa9", edgecolor="white", linewidth=1.0)
    kde = stats.gaussian_kde(df["value"])
    xs = np.linspace(df["value"].min(), df["value"].max(), 300)
    axes[0, 0].plot(xs, kde(xs), color="#0b1a7a", linewidth=1.8)
    axes[0, 0].set_title("Bieu do Histogram & Density Plot", fontsize=13, fontweight="bold")
    axes[0, 0].set_xlabel("Value")
    axes[0, 0].set_ylabel("Mat do (Density)")

    axes[0, 1].axis("off")
    stats_text = "\n".join([
        f"Rows: {len(df):,}",
        f"Mean: {df['value'].mean():.2f}",
        f"Std: {df['value'].std():.2f}",
        f"Skewness: {stats.skew(df['value']):.2f}",
        f"Min/Max: {df['value'].min():.2f}/{df['value'].max():.2f}",
        f"Anomaly points: {int(df['label'].sum()):,}",
    ])
    axes[0, 1].text(0.02, 0.9, stats_text, va="top", family="monospace", fontsize=12)

    plot_acf(df["value"], lags=288, ax=axes[1, 0], color="#4f79b5")
    axes[1, 0].set_title("ACF - Chu ky Ngay (Lags: 288)")
    axes[1, 0].set_xlabel("Do tre")
    axes[1, 0].set_ylabel("Tu tuong quan")

    plot_acf(df["value"], lags=2016, ax=axes[1, 1], color="#4f79b5")
    axes[1, 1].set_title("ACF - Chu ky Tuan (Lags: 2016)")
    axes[1, 1].set_xlabel("Do tre")
    axes[1, 1].set_ylabel("Tu tuong quan")

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_dir / "eda_template_vi.png", dpi=180)
    plt.close(fig)


plot_eda(df, CFG.output_dir)

### EDA image

![EDA](../artifacts/outputs/eda_template_vi.png)

## 4. Utilities

In [7]:
def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {
        "precision": p,
        "recall": r,
        "f1": f1,
        "true_positives": int(((y_pred == 1) & (y_true == 1)).sum()),
        "false_alarms": int(((y_pred == 1) & (y_true == 0)).sum()),
        "missed_anomalies": int(((y_pred == 0) & (y_true == 1)).sum()),
    }


def detect_iqr(series: pd.Series, window: int = 288, k: float = 2.0) -> pd.Series:
    q1 = series.rolling(window, min_periods=window // 4).quantile(0.25)
    q3 = series.rolling(window, min_periods=window // 4).quantile(0.75)
    iqr = q3 - q1
    return ((series < q1 - k * iqr) | (series > q3 + k * iqr)).fillna(False)


def detect_ewma(series: pd.Series, alpha: float = 0.1, threshold: float = 3.0) -> pd.Series:
    resid = series - series.ewm(alpha=alpha, adjust=False).mean()
    st = resid.rolling(288, min_periods=72).std().replace(0, np.nan)
    z = resid / st
    return (np.abs(z) > threshold).fillna(False)


def detect_log_zscore(series: pd.Series, window: int = 288, threshold: float = 3.0) -> pd.Series:
    s = np.log1p(series.clip(lower=0))
    mu = s.rolling(window, min_periods=window // 4).mean()
    sd = s.rolling(window, min_periods=window // 4).std().replace(0, np.nan)
    z = (s - mu) / sd
    return (np.abs(z) > threshold).fillna(False)

## 5. Feature Engineering & Isolation Forest

In [8]:
def build_features(df: pd.DataFrame, window: int = 12) -> pd.DataFrame:
    s = df["value"]
    ts = df["timestamp"]
    feats = pd.DataFrame(index=df.index)
    feats["value"] = s
    feats["rolling_mean_1h"] = s.rolling(window, min_periods=window).mean()
    feats["rolling_std_1h"] = s.rolling(window, min_periods=window).std()
    feats["rolling_mean_4h"] = s.rolling(window * 4, min_periods=window * 2).mean()
    feats["rate_of_change"] = s.diff()
    feats["rate_of_change_25m"] = s.diff(5)
    feats["lag_1"] = s.shift(1)
    feats["lag_1h"] = s.shift(window)
    feats["hour"] = ts.dt.hour
    feats["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)
    feats["z_score"] = (s - feats["rolling_mean_1h"]) / feats["rolling_std_1h"].replace(0, np.nan)
    return feats.replace([np.inf, -np.inf], np.nan).dropna()


features = build_features(df)
aligned = df.loc[features.index]
y_true = aligned["label"].astype(int).to_numpy()
X = StandardScaler().fit_transform(features)

stat_pred = detect_iqr(df["value"])
stat_metrics = evaluate(df["label"].astype(int).to_numpy(), stat_pred.astype(int).to_numpy())

rows = []
best = None
for c in [0.01, 0.02, 0.05]:
    model = IsolationForest(n_estimators=200, contamination=c, random_state=42, max_samples="auto")
    pred = (model.fit_predict(X) == -1).astype(int)
    metrics = evaluate(y_true, pred)
    row = {"detector": "Isolation Forest", "contamination": c, **metrics}
    rows.append(row)
    if best is None or row["f1"] > best["f1"]:
        best = {**row, "model": model, "y_pred": pred}

tuning_log = pd.DataFrame(rows)
tuning_log.to_csv(CFG.output_dir / "tuning_log.csv", index=False)
display(tuning_log)

,detector,contamination,precision,recall,f1,true_positives,false_alarms,missed_anomalies
0,Isolation Forest,0.01,0.837004,0.083774,0.152305,190,37,2078
1,Isolation Forest,0.02,0.872247,0.174603,0.290963,396,58,1872
2,Isolation Forest,0.05,0.597002,0.298501,0.398001,677,457,1591


## 6. Bonus Detectors

In [9]:
ewma_pred = detect_ewma(df["value"])
logz_pred = detect_log_zscore(df["value"])

bonus_rows = [
    {"detector": "Rolling IQR", "contamination": "", **stat_metrics},
    {"detector": "EWMA", "contamination": "", **evaluate(df["label"].astype(int).to_numpy(), ewma_pred.astype(int).to_numpy())},
    {"detector": "Log+3sigma", "contamination": "", **evaluate(df["label"].astype(int).to_numpy(), logz_pred.astype(int).to_numpy())},
    {k: v for k, v in best.items() if k not in {"model", "y_pred"}},
]

comparison = pd.DataFrame(bonus_rows)
comparison.to_csv(CFG.output_dir / "comparison.csv", index=False)
display(comparison)

,detector,contamination,precision,recall,f1,true_positives,false_alarms,missed_anomalies
0,Rolling IQR,,0.154517,0.109347,0.128066,248,1357,2020
1,EWMA,,0.182886,0.048060,0.076117,109,487,2159
2,Log+3sigma,,0.109467,0.048942,0.067642,111,903,2157
3,Isolation Forest,0.05,0.597002,0.298501,0.398001,677,457,1591


## 7. Save Model Artifact (.pkl, <1MB)

In [10]:
model_path = CFG.model_dir / "isolation_forest_machine_temperature.pkl"
with gzip.open(model_path, "wb") as f:
    pickle.dump(
        {
            "model": best["model"],
            "scaler": StandardScaler().fit(features),
            "feature_columns": list(features.columns),
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

size_bytes = model_path.stat().st_size
print("model:", model_path)
print("size bytes:", size_bytes, "(<1MB =", size_bytes < 1_048_576, ")")

model: D:\AWS\AIOPS\artifacts\models\isolation_forest_machine_temperature.pkl
size bytes: 624666 (<1MB = True )


## 8. Plot 2 Detector Results

In [11]:
def plot_detection(df: pd.DataFrame, stat_pred: pd.Series, if_pred: pd.Series, out_dir: Path) -> None:
    fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
    fig.suptitle("So sanh ket qua phat hien bat thuong", fontsize=16, fontweight="bold")

    for ax, pred, title in [
        (axes[0], stat_pred, "Detector thong ke: Rolling IQR"),
        (axes[1], if_pred, "Detector ML: Isolation Forest"),
    ]:
        ax.plot(df["timestamp"], df["value"], linewidth=0.9, color="#59636e", label="Metric goc")
        ax.scatter(df.loc[df["label"], "timestamp"], df.loc[df["label"], "value"], color="#1a9850", s=14, label="Ground truth NAB")
        ax.scatter(df.loc[pred, "timestamp"], df.loc[pred, "value"], color="#d73027", s=16, marker="x", label="Model detect")
        ax.set_title(title)
        ax.set_ylabel("Nhiet do may")
        ax.legend(loc="best")

    axes[1].set_xlabel("Thoi gian")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(out_dir / "detection_comparison_vi.png", dpi=180)
    plt.close(fig)


if_pred = pd.Series(False, index=df.index)
if_pred.loc[features.index] = best["y_pred"].astype(bool)
plot_detection(df, stat_pred, if_pred, CFG.output_dir)

### Detection image

![Detection](../artifacts/outputs/detection_comparison_vi.png)

## 9. Reflection

- Data type: univariate time series, skewed m?nh, c? failure windows/regime shift.
- Statistical method: Rolling IQR (ph? h?p skewed data h?n 3-sigma thu?n).
- ML method: Isolation Forest v?i feature engineering theo context th?i gian.
- Best detector theo F1 trong run hi?n t?i: Isolation Forest (contamination=0.05).
- Trade-off: contamination t?ng -> recall t?ng nh?ng false alarms t?ng.
- Production choice: d?ng IF l?m detector ch?nh, gi? IQR l?m baseline explainable.

In [12]:
# Experiments: EWMA (alpha=0.1), 3-sigma before/after log, multivariate IF vs univariate IF

def detect_3sigma(series: pd.Series, window: int = 288, threshold: float = 3.0) -> pd.Series:
    mu = series.rolling(window, min_periods=max(1, window // 4)).mean()
    sd = series.rolling(window, min_periods=max(1, window // 4)).std().replace(0, np.nan)
    z = (series - mu) / sd
    return (np.abs(z) > threshold).fillna(False)

# 1) EWMA detector (alpha=0.1)
ewma_pred_a01 = detect_ewma(df["value"], alpha=0.1, threshold=3.0)
ewma_metrics = evaluate(df["label"].astype(int).to_numpy(), ewma_pred_a01.astype(int).to_numpy())

# 2) 3-sigma on original series
three_orig = detect_3sigma(df["value"]) 
three_orig_metrics = evaluate(df["label"].astype(int).to_numpy(), three_orig.astype(int).to_numpy())

# 3) 3-sigma after log transform
s_log = np.log1p(df["value"].clip(lower=0))
three_log = detect_3sigma(s_log)
three_log_metrics = evaluate(df["label"].astype(int).to_numpy(), three_log.astype(int).to_numpy())

# 4) Multivariate: load a second NAB series and align on timestamp
def load_series_by_filename(filename: str) -> pd.DataFrame:
    csv_path = CFG.data_dir / filename
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing series file: {filename}")
    with (CFG.data_dir / "combined_windows.json").open("r", encoding="utf-8") as f:
        all_windows = json.load(f)
    # find matching key by suffix
    match_key = None
    for k in all_windows:
        if k.endswith(filename):
            match_key = k
            windows = all_windows[k]
            break
    if match_key is None:
        raise KeyError(f"No label windows found for {filename} in combined_windows.json")

    df2 = pd.read_csv(csv_path, parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
    df2["label"] = False
    for start, end in windows:
        mask = df2["timestamp"].between(pd.Timestamp(start), pd.Timestamp(end))
        df2.loc[mask, "label"] = True
    return df2

# choose a companion series (ambient temperature) if present
companion_file = "ambient_temperature_system_failure.csv"
if (CFG.data_dir / companion_file).exists():
    df2 = load_series_by_filename(companion_file)
    # align on timestamp
    merged = df[["timestamp", "value", "label"]].rename(columns={"value": "v1", "label": "label1"}).merge(
        df2[["timestamp", "value", "label"]].rename(columns={"value": "v2", "label": "label2"}), on="timestamp", how="inner"
    )
    merged["label_any"] = merged["label1"] | merged["label2"]

    # Multivariate IF using raw values (v1, v2)
    features_mv = merged[["v1", "v2"]].copy()
    X_mv = StandardScaler().fit_transform(features_mv)
    mv_model = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
    mv_pred = (mv_model.fit_predict(X_mv) == -1).astype(int)
    mv_metrics = evaluate(merged["label_any"].astype(int).to_numpy(), mv_pred)

    # Univariate IF on v1 aligned to merged index
    X_uni = StandardScaler().fit_transform(merged[["v1"]])
    uni_model = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
    uni_pred = (uni_model.fit_predict(X_uni) == -1).astype(int)
    uni_metrics = evaluate(merged["label1"].astype(int).to_numpy(), uni_pred)
else:
    merged = None
    mv_metrics = None
    uni_metrics = None

# Summarize results
rows = []
rows.append({"method": "EWMA (alpha=0.1)", **ewma_metrics})
rows.append({"method": "3-sigma (original)", **three_orig_metrics})
rows.append({"method": "3-sigma (log transform)", **three_log_metrics})
if mv_metrics is not None:
    rows.append({"method": "Univariate IF (aligned v1)", **uni_metrics})
    rows.append({"method": "Multivariate IF (v1+v2)", **mv_metrics})

results = pd.DataFrame(rows)
display(results)

# Print concise comparisons
print("-- Detailed comparison shown above --")
if mv_metrics is not None:
    print("Univariate IF vs Multivariate IF (f1):", uni_metrics["f1"], "->", mv_metrics["f1"])

# Save comparison artifact
results.to_csv(CFG.output_dir / "method_comparison_additional.csv", index=False)
print("Saved comparison to:", CFG.output_dir / "method_comparison_additional.csv")

,method,precision,recall,f1,true_positives,false_alarms,missed_anomalies
0,EWMA (alpha=0.1),0.182886,0.048060,0.076117,109,487,2159
1,3-sigma (original),0.115866,0.048942,0.068816,111,847,2157
2,3-sigma (log transform),0.109467,0.048942,0.067642,111,903,2157
3,Univariate IF (aligned v1),0.736842,0.368421,0.491228,70,25,120
4,Multivariate IF (v1+v2),0.736842,0.138614,0.233333,70,25,435


-- Detailed comparison shown above --
Univariate IF vs Multivariate IF (f1): 0.49122807017543857 -> 0.23333333333333334
Saved comparison to: D:\AWS\AIOPS\artifacts\outputs\method_comparison_additional.csv


In [13]:
# Bonus: Ensemble detector + report
# Aggregate detectors (Rolling IQR, EWMA alpha=0.1, Log-Z, IsolationForest) and evaluate ensemble

# Ensure required vars are present: df, features, best
if 'df' not in globals() or 'features' not in globals() or 'best' not in globals():
    raise RuntimeError('Required variables df, features, best must be present (run earlier cells)')

# prepare detectors
iqr_pred = detect_iqr(df['value'])
ewma_pred = detect_ewma(df['value'], alpha=0.1, threshold=3.0)
logz_pred = detect_log_zscore(df['value'])
if_pred = pd.Series(False, index=df.index)
if 'best' in globals() and isinstance(best, dict) and 'y_pred' in best:
    if_pred.loc[features.index] = best['y_pred'].astype(bool)

# combine into dataframe
det_df = pd.DataFrame({'IQR': iqr_pred.astype(int), 'EWMA': ewma_pred.astype(int), 'LogZ': logz_pred.astype(int), 'IF': if_pred.astype(int)}, index=df.index)

# majority vote (>=2 detectors)
det_df['ensemble_majority'] = (det_df.sum(axis=1) >= 2).astype(int)

ensemble_metrics = evaluate(df['label'].astype(int).to_numpy(), det_df['ensemble_majority'].to_numpy())

# compile metrics for each method
rows = []
for col in ['IQR','EWMA','LogZ','IF','ensemble_majority']:
    m = evaluate(df['label'].astype(int).to_numpy(), det_df[col].to_numpy())
    rows.append({'method': col, **m})

report = pd.DataFrame(rows)
display(report)

# Save report and detailed detector table
out_dir = CFG.output_dir
report.to_csv(out_dir / 'bonus_ensemble_report.csv', index=False)
det_df.to_csv(out_dir / 'bonus_detector_timeseries.csv', index=True)

# Plot a zoomed window around the first anomaly region for visualization
anomaly_idxs = df.index[df['label']].tolist()
if len(anomaly_idxs) > 0:
    start = max(0, anomaly_idxs[0] - 500)
    end = min(len(df)-1, anomaly_idxs[0] + 500)
else:
    start, end = 0, min(len(df)-1, 2000)

fig, ax = plt.subplots(1,1, figsize=(14,4))
ax.plot(df['timestamp'].iloc[start:end], df['value'].iloc[start:end], label='value', color='#59636e')
# overlay detectors
for name,color in [('IQR','#d73027'), ('EWMA','#f46d43'), ('LogZ','#fdae61'), ('IF','#4575b4'), ('ensemble_majority','#000000')]:
    mask = det_df[name].iloc[start:end].astype(bool)
    ax.scatter(df['timestamp'].iloc[start:end][mask], df['value'].iloc[start:end][mask], s=18, label=name, alpha=0.8)

ax.legend(loc='upper left')
ax.set_title('Bonus ensemble: detector overlay')
fig.tight_layout()
fig.savefig(out_dir / 'bonus_ensemble_overlay.png', dpi=150)
plt.close(fig)

print('Saved bonus report and artifacts to', out_dir)


,method,precision,recall,f1,true_positives,false_alarms,missed_anomalies
0,IQR,0.154517,0.109347,0.128066,248,1357,2020
1,EWMA,0.182886,0.048060,0.076117,109,487,2159
2,LogZ,0.109467,0.048942,0.067642,111,903,2157
3,IF,0.597002,0.298501,0.398001,677,457,1591
4,ensemble_majority,0.147788,0.073633,0.098293,167,963,2101


Saved bonus report and artifacts to D:\AWS\AIOPS\artifacts\outputs


### Bonus

Tóm tắt kết quả thử nghiệm (ghi vào phần Bonus để nộp):

- EWMA (α=0.1): precision=0.182886, recall=0.048060, f1=0.076117
- 3-sigma (original): precision=0.115866, recall=0.048942, f1=0.068816
- 3-sigma (log transform): precision=0.109467, recall=0.048942, f1=0.067642
- Univariate Isolation Forest (v1): precision=0.736842, recall=0.368421, f1=0.491228
- Multivariate Isolation Forest (v1+v2): precision=0.736842, recall=0.138614, f1=0.233333

Kết luận ngắn:
- `Univariate IF` đạt F1 tốt nhất trong cấu hình hiện tại.
- Log-transform không cải thiện 3-sigma cho bộ dữ liệu này.
- `Multivariate IF` giữ precision nhưng giảm mạnh recall → cần tinh chỉnh features/alignment/contamination.

Artifacts và file kết quả (đã lưu trong workspace):
- [method comparison thêm](artifacts/outputs/method_comparison_additional.csv)
- [bonus ensemble report](artifacts/outputs/bonus_ensemble_report.csv)
- [bonus detector timeseries (detailed)](artifacts/outputs/bonus_detector_timeseries.csv)
- [bonus ensemble overlay image](artifacts/outputs/bonus_ensemble_overlay.png)
- [saved model artifact](artifacts/models/isolation_forest_machine_temperature.pkl)

Ghi chú: nếu bạn muốn, tôi có thể tinh chỉnh contamination/feature set cho Multivariate IF hoặc thử các chiến lược ensemble khác và cập nhật phần Bonus này.